# Fine-tune Qwen2.5 for Support Ticket Classification

This notebook fine-tunes a small Qwen2.5 model to classify support tickets into 6 categories using **QLoRA** (memory-efficient) with **Unsloth** (2x faster).

**Before you start:** go to `Runtime -> Change runtime type` and select a **GPU** (T4 is free and works great).

**What you'll do:**
1. Install Unsloth
2. Upload your `train.jsonl` and `val.jsonl`
3. Load Qwen2.5 in 4-bit
4. Train the LoRA adapter (~5-15 min)
5. Test it
6. Download the result / export to Ollama

## 1. Install Unsloth

In [1]:
%%capture
# Unsloth installs everything it needs (transformers, peft, trl, bitsandbytes)
!pip install unsloth
# Keep to the latest nightly for newest model support
!pip install --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Upload your data
Run this cell, then use the file picker to upload **train.jsonl** and **val.jsonl** (the two files generated for you).

In [3]:
from google.colab import files
print('Upload train.jsonl and val.jsonl:')
uploaded = files.upload()
print('\nUploaded:', list(uploaded.keys()))

Upload train.jsonl and val.jsonl:


KeyboardInterrupt: 

## 3. Load Qwen2.5 in 4-bit
We use the 3B instruct model. To use a smaller/larger one, change `model_name` (e.g. `unsloth/Qwen2.5-1.5B-Instruct` or `unsloth/Qwen2.5-0.5B-Instruct`).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512   # ticket messages are short, so this is plenty

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-3B-Instruct',
    max_seq_length = max_seq_length,
    dtype = None,          # auto-detect
    load_in_4bit = True,   # QLoRA — fits comfortably on a free T4
)

# Attach LoRA adapters — we only train these (~1% of params)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # LoRA rank
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)
print('Model loaded and LoRA attached.')

## 4. Prepare the dataset
We load the JSONL files and format each example with Qwen's chat template.

In [ ]:
from datasets import load_dataset

def format_chat(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

train_ds = load_dataset('json', data_files='train.jsonl', split='train')
val_ds   = load_dataset('json', data_files='val.jsonl',   split='train')

# Fail early if the uploaded files do not match the expected chat format.
def validate_examples(ds, name):
    assert len(ds) > 0, f'{name} is empty'
    for i, ex in enumerate(ds):
        msgs = ex.get('messages')
        assert isinstance(msgs, list) and len(msgs) >= 3, (
            f'{name}[{i}] must contain at least system, user, and assistant messages')
        assert msgs[1].get('role') == 'user', f'{name}[{i}] message 2 must be user'
        assert msgs[2].get('role') == 'assistant', f'{name}[{i}] message 3 must be assistant'
        assert msgs[2].get('content', '').strip(), f'{name}[{i}] has an empty label'

validate_examples(train_ds, 'train')
validate_examples(val_ds, 'validation')

# Rebuild a deterministic, balanced split from both uploaded files.
# This gives every category 7 validation examples when possible.
from datasets import concatenate_datasets
all_ds = concatenate_datasets([train_ds, val_ds]).shuffle(seed=42)
labels = sorted({ex['messages'][2]['content'].strip() for ex in all_ds})
assert len(labels) == 6, f'Expected 6 labels, found {labels}'
val_indices = []
train_indices = []
for label in labels:
    indices = [i for i, ex in enumerate(all_ds)
               if ex['messages'][2]['content'].strip() == label]
    assert len(indices) >= 8, f'Not enough examples for {label}'
    val_indices.extend(indices[:7])
    train_indices.extend(indices[7:])
val_ds = all_ds.select(val_indices)
train_ds = all_ds.select(train_indices)

train_users = {ex['messages'][1]['content'].strip() for ex in train_ds}
val_users = {ex['messages'][1]['content'].strip() for ex in val_ds}
assert not train_users & val_users, 'Train/validation ticket overlap detected'

train_ds = train_ds.map(format_chat, batched=True)
val_ds   = val_ds.map(format_chat, batched=True)

print('Train examples:', len(train_ds))
print('Val examples:', len(val_ds))
print('\n--- One formatted example ---')
print(train_ds[0]['text'])

## 5. Train
This runs a few epochs. On a free T4 this takes roughly 5-15 minutes for this dataset size. Watch the loss go down.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # 3 passes over the data
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        eval_strategy = 'epoch',
        save_strategy = 'epoch',
        load_best_model_at_end = True,
        metric_for_best_model = 'eval_loss',
        greater_is_better = False,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)

trainer_stats = trainer.train()

## 6. Test your fine-tuned model
Try it on new messages it has never seen. It should reply with just the category.

In [ ]:
FastLanguageModel.for_inference(model)   # 2x faster inference
model.generation_config.max_length = None

SYSTEM = ('You are a support ticket classifier. Classify the customer message into '
          'exactly one category: Billing, Technical, Account, Shipping, '
          'Product Inquiry, or Cancellation. Respond with only the category name.')

def classify(message):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':message}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
        return_dict=True, return_attention_mask=True).to('cuda')
    out = model.generate(**inputs, max_new_tokens=8, use_cache=True,
                         temperature=0.0, do_sample=False)
    reply = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return reply.strip()

tests = [
    "I was double charged this month and need a refund",
    "The export button just spins and never finishes",
    "How do I reset my password? The email never comes",
    "Where is my package? It was due 3 days ago",
    "Does the Pro plan support more than 5 users?",
    "Please cancel my subscription before I get billed again",
]
for t in tests:
    print(f'{classify(t):18s} <- {t}')

## 7. Measure accuracy on the validation set

In [ ]:
import json
from collections import defaultdict

correct = 0
total = 0
by_label = defaultdict(lambda: {'correct': 0, 'total': 0})
for ex in val_ds:
    user_msg = ex['messages'][1]['content']
    gold = ex['messages'][2]['content']
    pred = classify(user_msg)
    pred_label = pred.split('\n')[0].strip().lower()
    gold_label = gold.strip().lower()
    total += 1
    by_label[gold_label]['total'] += 1
    if pred_label == gold_label:
        correct += 1
        by_label[gold_label]['correct'] += 1
    else:
        print(f'WRONG  gold={gold:16s} pred={pred:16s} | {user_msg[:50]}')

print(f'\nAccuracy: {correct}/{total} = {100*correct/total:.1f}%')
print('\nPer-category accuracy:')
for label, scores in sorted(by_label.items()):
    pct = 100 * scores['correct'] / scores['total']
    print(f"{label:18s} {scores['correct']:>3}/{scores['total']:<3} = {pct:5.1f}%")

## 8. Save and export

### Option A — download the LoRA adapter (small, few MB)

In [ ]:
model.save_pretrained('ticket_classifier_lora')
tokenizer.save_pretrained('ticket_classifier_lora')

# Zip it for download
!zip -r ticket_classifier_lora.zip ticket_classifier_lora
from google.colab import files
files.download('ticket_classifier_lora.zip')

### Option B — export to GGUF so you can run it in Ollama on your own PC
This merges the adapter into the base model and converts to GGUF (q4_k_m). The file is larger but runs directly in Ollama.

In [ ]:
# Saves a GGUF file you can load into Ollama
model.save_pretrained_gguf('ticket_classifier_gguf', tokenizer,
                           quantization_method='q4_k_m')
print('Done. Look in the ticket_classifier_gguf folder for the .gguf file.')

**To run the GGUF in Ollama** (on your PC after downloading it):

1. Create a file named `Modelfile` next to your `.gguf`:
   ```
   FROM ./ticket_classifier.gguf
   SYSTEM "You are a support ticket classifier. Classify the customer message into exactly one category: Billing, Technical, Account, Shipping, Product Inquiry, or Cancellation. Respond with only the category name."
   ```
2. Then run:
   ```
   ollama create ticket-classifier -f Modelfile
   ollama run ticket-classifier
   ```

That's it — your custom fine-tuned model now runs locally, and it works in the chat UI you built too!